In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.document_loaders import UnstructuredFileLoader
from langchain_core.documents import Document
from langchain.text_splitter import CharacterTextSplitter, SpacyTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.storage import LocalFileStore
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough
from langchain.memory import ConversationBufferMemory



# Declare LLM & Memory.
llm = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

memory = ConversationBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

# Setting the cache directory for embeddings.
cache_dir = LocalFileStore("./.cache/")

In [16]:
# Split & Vectorize the document.
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n\n',
    chunk_size=600,
    chunk_overlap=100
)


f = open("./files/document.txt")
content = f.read()

docs = splitter.split_text(content)
docs = [ Document(page_content=doc) for doc in docs ]
print(len(docs))

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

37


In [17]:
# Prepare retriver, prompt & function for loading chat history.
retriver = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. \
                Answer questions using only the following context. \
                If you don't know the answer just say you don't know, \
                don't make it up:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
        
    ]
)

def load_memory(_):
    return memory.load_memory_variables({})["history"]

# Implement the chain.
chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriver,
        "history": load_memory
    }
    | prompt
    | llm
)

In [18]:
# Function for invoking chain & saving the context on memory.
def invoke_chain(question):
    result = chain.invoke(question)
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

In [19]:
# Invoke the chain.
invoke_chain("Is Aaronson guilty?")

Yes, Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.content='Yes, Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.'


In [20]:
invoke_chain("What message did he write in the table?")
invoke_chain("Who is Julia?")

Winston wrote "2+2=5" in the dust on the table.content='Winston wrote "2+2=5" in the dust on the table.'
Julia is a woman who Winston falls in love with and has a romantic relationship with in the novel.content='Julia is a woman who Winston falls in love with and has a romantic relationship with in the novel.'


In [21]:
load_memory(_)

[HumanMessage(content='Is Aaronson guilty?'),
 AIMessage(content='Yes, Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.'),
 HumanMessage(content='What message did he write in the table?'),
 AIMessage(content='Winston wrote "2+2=5" in the dust on the table.'),
 HumanMessage(content='Who is Julia?'),
 AIMessage(content='Julia is a woman who Winston falls in love with and has a romantic relationship with in the novel.')]